# A-S Baseline — 30-Day Evaluation (Falces Marin Tables 2–5)

Runs the calibrated A-S agent on every day of DOGEUSDT data.  
Produces one table per metric with per-day rows and summary statistics.

In [1]:
import sys, pathlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

PROJECT_ROOT = pathlib.Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from procs.stochastic_processes.midprice_models import MarketReplayMidpriceModel
from procs.gym.helpers.fast_rollout import fast_simulate
from procs.gym.data_loader import load_multi_day
from procs.gym.calibration import tune_gamma
from procs.gym.reward_scale import estimate_reward_scale

%matplotlib inline


## 1. Load All Days

In [2]:
DATA_DIR = r"C:\Users\john-\Documents\Thesis_AI4T\datasets"
daily_S, daily_dt, dates = load_multi_day(DATA_DIR, pair="DOGEUSDT")

  2025-01-01:  713,815 snapshots, σ=0.000021
  2025-01-02:  766,464 snapshots, σ=0.000035
  2025-01-03:  776,383 snapshots, σ=0.000047
  2025-01-04:  778,293 snapshots, σ=0.000045
  2025-01-05:  723,494 snapshots, σ=0.000031
  2025-01-06:  766,311 snapshots, σ=0.000035
  2025-01-07:  787,093 snapshots, σ=0.000062
  2025-01-08:  821,589 snapshots, σ=0.000052
  2025-01-09:  809,421 snapshots, σ=0.000046
  2025-01-10:  789,320 snapshots, σ=0.000045
  2025-01-11:  724,826 snapshots, σ=0.000023
  2025-01-12:  719,550 snapshots, σ=0.000022
  2025-01-13:  819,981 snapshots, σ=0.000046
  2025-01-14:  775,454 snapshots, σ=0.000038
  2025-01-15:  782,299 snapshots, σ=0.000047
  2025-01-16:  788,946 snapshots, σ=0.000048
  2025-01-17:  801,259 snapshots, σ=0.000061
  2025-01-18:  829,125 snapshots, σ=0.000074
  2025-01-19:  842,892 snapshots, σ=0.000096
  2025-01-20:  857,508 snapshots, σ=0.000115
  2025-01-21:  848,107 snapshots, σ=0.000112
  2025-01-22:  812,180 snapshots, σ=0.000044
  2025-01-

## 2. Parameters

In [3]:
def calibrate_from_arrays(
    S: np.ndarray,
    dt: np.ndarray,
    tick_size: float = 0.00001,
    n_depth_ticks: int = 5,
    min_arrivals: int = 10,
) -> tuple[float, float, float]:
    """
    Calibrate σ, A, κ from already-loaded midprice and dt arrays.

    Implements the hftbacktest GLFT calibration procedure:
        - σ: median of rolling 10-min window estimates (arithmetic, A-S convention)
        - A, κ: OLS on log λ(δ) = log A − κ·δ fitted to the near-mid
                depth range (first n_depth_ticks tick levels)

    Market order arrivals are inferred from best-bid/ask changes between
    consecutive L2 snapshots — identical to hftbacktest's
    measure_trading_intensity, adapted for pre-loaded arrays.

    Parameters
    ----------
    S : np.ndarray       Midprice time series (price units)
    dt : np.ndarray      Time deltas in seconds (same length as S)
    tick_size : float    Minimum price increment
    n_depth_ticks : int  Number of tick levels to use in regression
    min_arrivals : int   Minimum arrivals per bin for inclusion in fit

    Returns
    -------
    sigma, A, kappa : float
    """
    T = float(dt.sum())
    N = len(S)

    # ── σ: arithmetic volatility (A-S convention: dS = σ dW) ─
    dS = np.diff(S)
    dt_mid = dt[1:]
    window_size = max(1, int(600.0 / np.median(dt_mid[dt_mid > 0])))
    sigma_estimates = []
    for start in range(0, N - 1 - window_size, window_size):
        dS_w = dS[start:start + window_size]
        dt_w = dt_mid[start:start + window_size]
        total_t = dt_w.sum()
        if total_t > 0:
            sigma_estimates.append(np.sqrt(np.sum(dS_w**2) / total_t))
    sigma = float(np.median(sigma_estimates)) if sigma_estimates else float(
        np.sqrt(np.sum(dS**2) / T)
    )

    # ── Detect market order arrivals from L2 snapshot changes ─
    # We only have best bid/ask from midprice — reconstruct from S and tick
    # Since midprice = (ask + bid)/2 and spread >= tick:
    #   best_ask ≈ S + tick/2,  best_bid ≈ S - tick/2
    # Arrival detection: consecutive midprice jumps ≥ tick signal a trade.
    mid_diff = np.abs(np.diff(S))
    arrival_mask = mid_diff >= tick_size * 0.5
    arrival_depths = mid_diff[arrival_mask]   # depth ≈ |ΔS| proxy

    # ── Histogram: λ(δ) per tick bin ─────────────────────────
    fit_depth_max = n_depth_ticks * tick_size
    bin_edges = np.arange(0, n_depth_ticks + 1) * tick_size
    bin_centres = (bin_edges[:-1] + bin_edges[1:]) / 2

    counts, _ = np.histogram(arrival_depths, bins=bin_edges)
    lambda_emp = counts / T

    valid = counts >= min_arrivals
    if valid.sum() < 2:
        # Fallback: use simple mean arrival rate and a default κ
        A_fallback = float(len(arrival_depths) / T)
        return sigma, A_fallback, 35_000.0

    # ── OLS: log λ = log A − κ·δ ─────────────────────────────
    x = bin_centres[valid]
    y = np.log(lambda_emp[valid])
    n = len(x)
    sx = x.sum(); sy = y.sum()
    sx2 = (x**2).sum(); sxy = (x * y).sum()
    slope = (n * sxy - sx * sy) / (n * sx2 - sx**2)
    intercept = (sy - slope * sx) / n

    kappa = float(-slope)
    A = float(np.exp(intercept))

    # Sanity clamp: κ should be positive; if fit fails, use prior
    if kappa <= 0 or not np.isfinite(kappa):
        kappa = 35_000.0
    if A <= 0 or not np.isfinite(A):
        A = 0.8

    return sigma, A, kappa


In [ ]:
# Fixed parameters (same across all days)
tick_size = 0.00001
Q_MAX     = 10
phi       = 0.01
N_traj    = 20     # trajectories per day for metric averaging
seed      = 42

## 3. Run A-S on Every Day

In [7]:
results = []

for i, (S, dt, date) in enumerate(zip(daily_S, daily_dt, dates)):
    T = float(dt.sum())

    # ── 1. Calibrate σ, A, κ from this day's data ────────────
    sigma, A, kappa = calibrate_from_arrays(S, dt, tick_size=tick_size)

    # ── 2. Tune γ for this day via Optuna ─────────────────────
    # gamma_range narrowed from theory: γ ≥ tick / (σ²·T) ≈ 0.1 for DOGE.
    # n_trials=20 is the minimum for TPE to outperform random search.
    # Suppress Optuna output — we print our own summary below.
    best_gamma, _ = tune_gamma(
        midprices=S, dt_array=dt,
        sigma=sigma, kappa=kappa, A=A,
        tick_size=tick_size, Q_MAX=Q_MAX,
        gamma_range=(0.0001, 1),
        n_trials=20,
        num_trajectories=N_traj,
        seed=seed,
        verbose=False,
    )

    # ── 3. Evaluate A-S agent with calibrated parameters ──────
    stats = fast_simulate(
        midprices=S, dt_array=dt,
        gamma=best_gamma, sigma=sigma, kappa=kappa, A=A,
        terminal_time=T, tick_size=tick_size, Q_MAX=Q_MAX,
        num_trajectories=N_traj, seed=seed,
        use_linear_approximation=False,
    )

    results.append({
        "Day":        date,
        "Sharpe":     stats["sharpe"].mean(),
        "Sortino":    stats["sortino"].mean(),
        "Max DD":     stats["max_drawdown"].mean(),
        "P&L-to-MAP": stats["pnl_to_map"].mean(),
        "Final PnL":  stats["total_pnl"].mean(),
        "Mean |q|":   stats["mean_abs_q"].mean(),
        "sigma":      sigma,
        "A":          A,
        "kappa":      kappa,
        "gamma":      best_gamma,
    })

    print(f"  [{i+1:2d}/30] {date}  γ={best_gamma:.4f}  σ={sigma:.6f}  "
          f"κ={kappa:.0f}  A={A:.3f}  "
          f"Sharpe={stats['sharpe'].mean():+.4f}  "
          f"MaxDD={stats['max_drawdown'].mean():.4f}  "
          f"PnL={stats['total_pnl'].mean():+.4f}")

df = pd.DataFrame(results).set_index("Day")
print(f"\nDone: {len(df)} days.")


  [ 1/30] 2025-01-01  γ=0.9769  σ=0.000020  κ=39822  A=0.256  Sharpe=+0.0455  MaxDD=0.0046  PnL=+0.3342
  [ 2/30] 2025-01-02  γ=0.9840  σ=0.000027  κ=21646  A=0.279  Sharpe=+0.0620  MaxDD=0.0046  PnL=+0.6808
  [ 3/30] 2025-01-03  γ=0.9658  σ=0.000036  κ=11017  A=0.248  Sharpe=+0.0579  MaxDD=0.0101  PnL=+1.2057
  [ 4/30] 2025-01-04  γ=0.9956  σ=0.000036  κ=6184  A=0.197  Sharpe=+0.0685  MaxDD=0.0161  PnL=+1.8378
  [ 5/30] 2025-01-05  γ=0.9742  σ=0.000027  κ=13892  A=0.173  Sharpe=+0.0529  MaxDD=0.0088  PnL=+0.6985
  [ 6/30] 2025-01-06  γ=0.9760  σ=0.000028  κ=14043  A=0.221  Sharpe=+0.0611  MaxDD=0.0072  PnL=+0.8756
  [ 7/30] 2025-01-07  γ=0.9703  σ=0.000037  κ=4553  A=0.238  Sharpe=+0.0752  MaxDD=0.0179  PnL=+3.0653
  [ 8/30] 2025-01-08  γ=0.9712  σ=0.000043  κ=7812  A=0.333  Sharpe=+0.0911  MaxDD=0.0087  PnL=+2.2962
  [ 9/30] 2025-01-09  γ=0.9220  σ=0.000039  κ=11253  A=0.356  Sharpe=+0.0835  MaxDD=0.0068  PnL=+1.6607
  [10/30] 2025-01-10  γ=0.9945  σ=0.000032  κ=14017  A=0.331  Sharp

[W 2026-04-12 22:59:55,294] Trial 16 failed with parameters: {'gamma': 0.18841194038575612} because of the following error: KeyboardInterrupt().
Traceback (most recent call last):
  File "c:\Users\john-\anaconda3\envs\mysimenv\Lib\site-packages\optuna\study\_optimize.py", line 206, in _run_trial
    value_or_values = func(trial)
                      ^^^^^^^^^^^
  File "c:\Users\john-\Documents\Thesis_AI4T\procs\gym\calibration.py", line 337, in objective
    stats = fast_simulate(
            ^^^^^^^^^^^^^^
  File "c:\Users\john-\Documents\Thesis_AI4T\procs\gym\helpers\fast_rollout.py", line 169, in fast_simulate
    u_fill = rng.uniform(size=(N, 2))
             ^^^^^^^^^^^^^^^^^^^^^^^^
KeyboardInterrupt
[W 2026-04-12 22:59:55,294] Trial 16 failed with value None.


KeyboardInterrupt: 

## 4. Full Summary Table

In [ ]:
summary = pd.DataFrame({
    col: [df[col].mean(), df[col].std(), df[col].median()]
    for col in df.columns
}, index=["Mean", "Std", "Median"])

full = pd.concat([df, summary])
print(full.to_string(float_format="%.6f"))

## 5. Per-Metric Tables (Falces Marin style)

In [ ]:
for metric in ["Sharpe", "Sortino", "Max DD", "P&L-to-MAP", "Final PnL"]:
    col = df[[metric]]
    s = pd.DataFrame({
        metric: [col[metric].mean(), col[metric].std(), col[metric].median()]
    }, index=["Mean", "Std", "Median"])
    table = pd.concat([col, s])
    print(f"\n{'='*50}")
    print(f"  {metric}")
    print(f"{'='*50}")
    print(table.to_string(float_format="%.6f"))

## 6. Bar Charts

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

metrics = ["Sharpe", "Sortino", "Max DD", "P&L-to-MAP", "Final PnL", "sigma"]
colors = ["steelblue", "seagreen", "indianred", "mediumpurple", "darkorange", "grey"]

for ax, metric, color in zip(axes.flat, metrics, colors):
    ax.bar(range(len(df)), df[metric], color=color, alpha=0.8)
    ax.axhline(y=df[metric].mean(), color="k", ls="--", lw=1,
               label=f"mean={df[metric].mean():.4f}")
    ax.set_title(metric)
    ax.set_xlabel("Day")
    ax.set_ylabel(metric)
    ax.set_xticks(range(len(df)))
    ax.set_xticklabels(df.index, rotation=45, ha="right", fontsize=7)
    ax.legend(fontsize=8)

plt.suptitle(f"A-S Baseline (per-day calibration) - {len(df)} Days DOGEUSDT", fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
df.to_csv("C:/Users/john-/Documents/Thesis_AI4T/modelsas_baseline_30day_results.csv")
print("Saved to models/as_baseline_30day_results.csv")